# WSI DeID Prototype Notebook

This notebook is a first test-bed for **whole-slide image de-identification / pseudonymisation**.

It focuses on the practical WSI problem: labels, barcodes, macro images, and handwritten identifiers near the non-barcode end of the glass slide.

This is not a guarantee of legal anonymisation. If a re-identification key is retained, the workflow should be described as **pseudonymisation / de-identification**.

## Install dependencies

Run this cell if the imports fail. OpenSlide also needs the native OpenSlide library installed on the system.

In [ ]:
# Optional install cell
# %pip install numpy pillow matplotlib opencv-python openslide-python

In [ ]:
from pathlib import Path
import csv
import hashlib
import secrets
from datetime import datetime

import numpy as np
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

try:
    import cv2
except ImportError as exc:
    raise ImportError('Install opencv-python: %pip install opencv-python') from exc

try:
    import openslide
except ImportError:
    openslide = None

OUT_DIR = Path('wsi_deid_qc')
OUT_DIR.mkdir(exist_ok=True)
print('QC output folder:', OUT_DIR.resolve())

## Configuration

For a standard glass slide, the physical dimensions are approximately **75 mm x 25 mm**. The script masks a configurable number of millimetres from the barcode/label end and from the opposite handwritten end.

In [ ]:
SLIDE_LENGTH_MM = 75.0
SLIDE_WIDTH_MM = 25.0

# Conservative defaults. Adjust after reviewing QC thumbnails.
MASK_BARCODE_END_MM = 22.0
MASK_HANDWRITTEN_END_MM = 12.0
MASK_TOP_BOTTOM_MARGIN_MM = 1.0

# Supported via OpenSlide in many installations:
WSI_EXTENSIONS = {
    '.svs', '.tif', '.tiff', '.ndpi', '.vms', '.vmu', '.scn', '.mrxs', '.dcm', '.bif', '.svslide'
}

print('Configured WSI extensions:', ', '.join(sorted(WSI_EXTENSIONS)))

## Pseudonymous ID and key helpers

The key file should be stored separately from de-identified slides, ideally encrypted and access-controlled.

In [ ]:
def make_study_id(prefix='WSI'):
    return f'{prefix}_{secrets.token_hex(4).upper()}'


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def append_key_row(key_csv, original_path, study_id, notes=''):
    key_csv = Path(key_csv)
    is_new = not key_csv.exists()
    with key_csv.open('a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(
            f,
            fieldnames=['study_id', 'original_filename', 'original_path', 'sha256', 'created_at', 'notes'],
        )
        if is_new:
            writer.writeheader()
        writer.writerow({
            'study_id': study_id,
            'original_filename': Path(original_path).name,
            'original_path': str(Path(original_path).resolve()),
            'sha256': sha256_file(original_path) if Path(original_path).is_file() else '',
            'created_at': datetime.now().isoformat(timespec='seconds'),
            'notes': notes,
        })

## Synthetic macro image

This creates a fake whole-slide macro image with tissue, a barcode label, and handwritten numbers at the opposite end. It allows testing the redaction logic before using patient data.

In [ ]:
def create_synthetic_macro(width=1500, height=500, barcode_side='left'):
    img = Image.new('RGB', (width, height), (235, 239, 238))
    draw = ImageDraw.Draw(img)

    # Glass slide body
    slide_box = (45, 55, width - 45, height - 55)
    draw.rounded_rectangle(slide_box, radius=16, fill=(218, 229, 229), outline=(160, 175, 175), width=3)

    label_w = int(width * 0.23)
    if barcode_side == 'left':
        label_box = (70, 85, 70 + label_w, height - 85)
        scratch_x = width - 250
    else:
        label_box = (width - 70 - label_w, 85, width - 70, height - 85)
        scratch_x = 125

    # Paper label with barcode-like bars
    draw.rectangle(label_box, fill=(247, 245, 224), outline=(80, 80, 80), width=2)
    x0, y0, x1, y1 = label_box
    for i, x in enumerate(range(x0 + 22, x1 - 22, 12)):
        bar_w = 3 if i % 3 else 7
        draw.rectangle((x, y0 + 55, x + bar_w, y1 - 70), fill=(35, 35, 35))
    draw.text((x0 + 22, y0 + 18), 'CASE 24-12345', fill=(20, 20, 20))

    # Tissue-like regions
    rng = np.random.default_rng(7)
    for _ in range(14):
        cx = int(rng.integers(int(width * 0.34), int(width * 0.72)))
        cy = int(rng.integers(150, height - 150))
        rx = int(rng.integers(45, 105))
        ry = int(rng.integers(18, 58))
        color = tuple(int(v) for v in rng.integers([135, 60, 115], [210, 140, 190]))
        draw.ellipse((cx - rx, cy - ry, cx + rx, cy + ry), fill=color)

    # Scratch/handwritten identifier near the non-barcode end
    draw.text((scratch_x, 165), 'A17 / 6', fill=(35, 35, 35))
    draw.line((scratch_x, 210, scratch_x + 135, 185, scratch_x + 185, 230), fill=(40, 40, 40), width=4)
    draw.line((scratch_x + 20, 245, scratch_x + 155, 260), fill=(45, 45, 45), width=3)
    return img


synthetic = create_synthetic_macro(barcode_side='left')
plt.figure(figsize=(12, 4))
plt.imshow(synthetic)
plt.axis('off');

## Redaction logic

The prototype uses a deliberately conservative approach:

- find the slide rectangle in the macro image
- infer the barcode/label side by looking for dense dark vertical barcode-like texture near either end
- mask the barcode/label end
- mask the opposite handwritten/scratch end
- save QC images

In [ ]:
def find_slide_box(rgb_img):
    arr = np.array(rgb_img.convert('RGB'))
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    # Slide/glass is darker than the outside background in most macro images.
    _, mask = cv2.threshold(gray, 248, 255, cv2.THRESH_BINARY_INV)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((25, 25), np.uint8))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return (0, 0, arr.shape[1], arr.shape[0])
    contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(contour)
    return (x, y, x + w, y + h)


def infer_barcode_side(rgb_img, slide_box):
    arr = np.array(rgb_img.convert('RGB'))
    x0, y0, x1, y1 = slide_box
    W = x1 - x0
    end_w = max(1, int(W * 0.28))
    left = arr[y0:y1, x0:x0 + end_w]
    right = arr[y0:y1, x1 - end_w:x1]

    def barcode_score(region):
        gray = cv2.cvtColor(region, cv2.COLOR_RGB2GRAY)
        dark = gray < 90
        edges = cv2.Canny(gray, 40, 140)
        vertical_kernel = np.ones((25, 3), np.uint8)
        vertical = cv2.morphologyEx(edges, cv2.MORPH_OPEN, vertical_kernel)
        return float(dark.mean()) + 2.0 * float((vertical > 0).mean())

    left_score = barcode_score(left)
    right_score = barcode_score(right)
    return 'left' if left_score >= right_score else 'right', left_score, right_score


def redact_slide_ends(
    rgb_img,
    barcode_side='auto',
    slide_length_mm=75.0,
    barcode_end_mm=22.0,
    handwritten_end_mm=12.0,
    top_bottom_margin_mm=1.0,
    fill=(255, 255, 255),
):
    img = rgb_img.convert('RGB')
    arr = np.array(img).copy()
    mask = np.zeros(arr.shape[:2], dtype=np.uint8)
    box = find_slide_box(img)
    x0, y0, x1, y1 = box
    slide_px = x1 - x0

    if barcode_side == 'auto':
        barcode_side, left_score, right_score = infer_barcode_side(img, box)
    else:
        left_score = right_score = None

    barcode_px = int(slide_px * barcode_end_mm / slide_length_mm)
    handwritten_px = int(slide_px * handwritten_end_mm / slide_length_mm)
    margin_px = int((y1 - y0) * top_bottom_margin_mm / SLIDE_WIDTH_MM)
    yy0 = max(0, y0 + margin_px)
    yy1 = min(arr.shape[0], y1 - margin_px)

    if barcode_side == 'left':
        mask[yy0:yy1, x0:min(x1, x0 + barcode_px)] = 255
        mask[yy0:yy1, max(x0, x1 - handwritten_px):x1] = 255
    else:
        mask[yy0:yy1, max(x0, x1 - barcode_px):x1] = 255
        mask[yy0:yy1, x0:min(x1, x0 + handwritten_px)] = 255

    arr[mask > 0] = fill
    info = {
        'slide_box': box,
        'barcode_side': barcode_side,
        'left_score': left_score,
        'right_score': right_score,
        'barcode_px': barcode_px,
        'handwritten_px': handwritten_px,
    }
    return Image.fromarray(arr), Image.fromarray(mask), info

In [ ]:
redacted, mask, info = redact_slide_ends(synthetic)
print(info)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(synthetic)
axes[0].set_title('Before')
axes[1].imshow(mask, cmap='gray')
axes[1].set_title('Mask')
axes[2].imshow(redacted)
axes[2].set_title('After')
for ax in axes:
    ax.axis('off')
plt.tight_layout()

synthetic.save(OUT_DIR / 'synthetic_before.png')
mask.save(OUT_DIR / 'synthetic_mask.png')
redacted.save(OUT_DIR / 'synthetic_after.png')

## Optional: test on one real WSI file

This does not modify the original WSI. It extracts associated images for QC. If an associated image is named `label`, it is fully blanked. If it is named `macro` or `overview`, slide-end masking is applied.

In [ ]:
WSI_PATH = Path(r'')  # Example: Path(r'D:\slides\case_001.ndpi')


def process_associated_images_for_qc(wsi_path, study_id=None, output_dir=OUT_DIR):
    if openslide is None:
        raise RuntimeError('openslide-python is not installed. Run: %pip install openslide-python')
    wsi_path = Path(wsi_path)
    study_id = study_id or make_study_id()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    slide = openslide.OpenSlide(str(wsi_path))
    print('Associated images:', list(slide.associated_images.keys()))
    print('Metadata properties:', len(slide.properties))

    metadata_report = output_dir / f'{study_id}_metadata_keys.txt'
    with metadata_report.open('w', encoding='utf-8') as f:
        for key, value in sorted(slide.properties.items()):
            f.write(f'{key}: {value}\n')

    for name, assoc in slide.associated_images.items():
        assoc_rgb = assoc.convert('RGB')
        safe_name = name.lower().replace(' ', '_')
        before_path = output_dir / f'{study_id}_{safe_name}_before.png'
        after_path = output_dir / f'{study_id}_{safe_name}_after.png'
        mask_path = output_dir / f'{study_id}_{safe_name}_mask.png'
        assoc_rgb.save(before_path)

        if safe_name == 'label':
            after = Image.new('RGB', assoc_rgb.size, (255, 255, 255))
            mask_img = Image.new('L', assoc_rgb.size, 255)
        elif safe_name in {'macro', 'overview'}:
            after, mask_img, info = redact_slide_ends(
                assoc_rgb,
                barcode_side='auto',
                barcode_end_mm=MASK_BARCODE_END_MM,
                handwritten_end_mm=MASK_HANDWRITTEN_END_MM,
            )
            print(safe_name, info)
        else:
            after = assoc_rgb
            mask_img = Image.new('L', assoc_rgb.size, 0)

        after.save(after_path)
        mask_img.save(mask_path)

    slide.close()
    return study_id


if WSI_PATH.exists():
    sid = process_associated_images_for_qc(WSI_PATH)
    print('Processed study ID:', sid)
else:
    print('Set WSI_PATH to a real .ndpi/.svs/.scn/.mrxs/.tif/.dcm file and run again.')

## Optional: batch QC folder

This creates pseudonymous IDs, appends a key CSV, and writes QC images for each supported WSI file. It still does not rewrite or modify the original WSI files.

In [ ]:
INPUT_DIR = Path(r'')  # Example: Path(r'D:\slides')
KEY_CSV = OUT_DIR / 'pseudonym_key.csv'


def iter_wsi_files(input_dir):
    input_dir = Path(input_dir)
    for path in input_dir.rglob('*'):
        if path.is_file() and path.suffix.lower() in WSI_EXTENSIONS:
            yield path


def batch_qc(input_dir, key_csv=KEY_CSV):
    files = list(iter_wsi_files(input_dir))
    print('Found WSI files:', len(files))
    for path in files:
        study_id = make_study_id()
        print('Processing', path.name, '->', study_id)
        append_key_row(key_csv, path, study_id, notes='QC prototype only; original WSI not modified')
        try:
            process_associated_images_for_qc(path, study_id=study_id, output_dir=OUT_DIR)
        except Exception as exc:
            print('FAILED:', path, exc)


if INPUT_DIR.exists():
    batch_qc(INPUT_DIR)
else:
    print('Set INPUT_DIR to a folder containing WSI files and run again.')

## Next development steps

1. Test this on 20-50 non-shared internal slides and review before/after QC images.
2. Tune `MASK_BARCODE_END_MM` and `MASK_HANDWRITTEN_END_MM` for local scanner output.
3. Add metadata redaction rules by scanner/vendor.
4. Add clean derivative export to OME-TIFF or DICOM-WSI.
5. Add encrypted key storage and a formal QC report.